# Colab Bootstrap — StockVolatilitySight full pipeline

**What this notebook does:**

1. Mounts your Google Drive (expects `MyDrive/StockVolatilitySight/data/raw/` with `aaii_sentiment.xls` + optional `SPY_ohlcv.parquet`, `vix_family.parquet`).
2. Clones the private GitHub repo using a fine-grained PAT (entered at runtime via `getpass` — never written to the notebook output).
3. Installs `requirements.txt`.
4. Copies raw inputs from Drive into the cloned repo's `data/raw/`.
5. Executes nb 01 → nb 07 sequentially via `nbconvert`. After each notebook, syncs `models/`, `data/processed/`, `supplementary/figures/`, and the executed notebook itself back to Drive (so a Colab disconnect doesn't lose hours of training).

**Total wall time:** ~3-4 hours on Colab free-tier (no GPU). The training scripts explicitly use CPU — GPU would need a code change and introduces non-determinism.

**Disconnect resilience:** each notebook has its own cell below. If Colab disconnects mid-nb05, you re-run the setup cells (they're idempotent — skip if already done) and then just the nb05 + nb06 + nb07 cells.

## Prerequisites (do these ONCE before running)

1. Google Drive folder layout:
   ```
   MyDrive/
   └── StockVolatilitySight/
       └── data/
           └── raw/
               ├── aaii_sentiment.xls       ← REQUIRED
               ├── SPY_ohlcv.parquet        ← optional (speeds up nb01)
               └── vix_family.parquet       ← optional (speeds up nb01)
   ```
2. GitHub fine-grained Personal Access Token with Contents: Read for this repo.


## 1. Configuration — edit if needed

Defaults below match the standard setup. Change `DRIVE_PROJECT_DIR` if your Drive folder is named differently; change `GITHUB_REPO` if you forked.


In [ ]:
import os
from pathlib import Path

GITHUB_REPO         = "maharajhaider/StockVolatilitySight"
REPO_BRANCH         = "main"
DRIVE_PROJECT_DIR   = "StockVolatilitySight"    # under /content/drive/MyDrive/
COLAB_REPO_ROOT     = Path("/content/repo")
DRIVE_MOUNT_POINT   = Path("/content/drive")
DRIVE_PROJECT_ROOT  = DRIVE_MOUNT_POINT / "MyDrive" / DRIVE_PROJECT_DIR

print(f"Repo               : {GITHUB_REPO} (branch: {REPO_BRANCH})")
print(f"Colab clone target : {COLAB_REPO_ROOT}")
print(f"Drive project root : {DRIVE_PROJECT_ROOT}")


## 2. Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount(str(DRIVE_MOUNT_POINT))

assert DRIVE_PROJECT_ROOT.exists(), (
    f"Drive folder not found at {DRIVE_PROJECT_ROOT}. "
    f"Check the folder name in cell 1 above matches your Drive layout."
)

raw_dir = DRIVE_PROJECT_ROOT / "data" / "raw"
assert (raw_dir / "aaii_sentiment.xls").exists(), (
    f"Required file missing: {raw_dir / 'aaii_sentiment.xls'}. "
    "Upload it to Drive before running."
)
print("Drive mounted and raw/aaii_sentiment.xls verified.")
for f in sorted(raw_dir.iterdir()):
    print(f"  raw/{f.name}  ({f.stat().st_size / 1024:.0f} KB)")


## 3. Clone the private repo

Prompts you for a GitHub PAT (masked via `getpass`). The token is used only to assemble the HTTPS clone URL and is discarded after.


In [ ]:
import getpass, shutil, subprocess

if COLAB_REPO_ROOT.exists():
    print(f"{COLAB_REPO_ROOT} already exists — removing and re-cloning for a clean state.")
    shutil.rmtree(COLAB_REPO_ROOT)

github_pat = getpass.getpass("GitHub fine-grained PAT (github_pat_...): ").strip()
if not github_pat:
    raise RuntimeError("No PAT provided — clone would fail on private repo.")

clone_url = f"https://{github_pat}@github.com/{GITHUB_REPO}.git"
result = subprocess.run(
    ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", clone_url, str(COLAB_REPO_ROOT)],
    capture_output=True, text=True,
)

# Scrub token from any printed URL on failure
if result.returncode != 0:
    err = result.stderr.replace(github_pat, "<PAT>")
    raise RuntimeError(f"git clone failed: {err}")

print(f"Cloned into {COLAB_REPO_ROOT}")
print(subprocess.run(["git", "-C", str(COLAB_REPO_ROOT), "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

# Forget the PAT in this cell's memory (doesn't affect the cloned repo; git stores
# the remote without the token since we used a one-shot URL).
del github_pat, clone_url


## 4. Install Python requirements

Colab's base image already has most of what we need; `pip install -r requirements.txt` tops it up. `-q` for quiet output; bump to `-v` if debugging.


In [ ]:
os.chdir(COLAB_REPO_ROOT)

result = subprocess.run(
    ["pip", "install", "-q", "-r", "requirements.txt"],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
    raise RuntimeError(f"pip install failed (rc={result.returncode})")
print("Requirements installed.")


## 5. Copy raw inputs from Drive into the cloned repo

Copies `aaii_sentiment.xls` (required) and any cached parquets (optional). Skips processed artifacts — nb 01 regenerates those.


In [ ]:
repo_raw = COLAB_REPO_ROOT / "data" / "raw"
repo_raw.mkdir(parents=True, exist_ok=True)

for src_file in sorted(raw_dir.iterdir()):
    if src_file.is_file():
        shutil.copy2(src_file, repo_raw / src_file.name)
        print(f"  copied {src_file.name}  ({src_file.stat().st_size / 1024:.0f} KB)")

print("\nRaw inputs ready at", repo_raw)


## 6. Notebook-execution and Drive-sync helpers

`run_notebook()` invokes `jupyter nbconvert --execute` on a given notebook with a per-notebook timeout (passed as `timeout_sec`). `sync_to_drive()` rsyncs the output directories back to Drive so a disconnect mid-pipeline doesn't lose earlier progress.


In [ ]:
import time

def run_notebook(nb_filename: str, timeout_sec: int = 1800):
    """Execute notebooks/<nb_filename> via nbconvert, stream logs live."""
    nb_path = COLAB_REPO_ROOT / "notebooks" / nb_filename
    assert nb_path.exists(), f"Missing: {nb_path}"

    print("=" * 80)
    print(f"  Running {nb_filename}  (timeout {timeout_sec}s = {timeout_sec // 60} min)")
    print("=" * 80)
    t0 = time.time()

    cmd = [
        "jupyter", "nbconvert", "--to", "notebook", "--execute", "--inplace",
        f"--ExecutePreprocessor.timeout={timeout_sec}",
        str(nb_path),
    ]
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        cwd=str(COLAB_REPO_ROOT), text=True, bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
    elapsed = time.time() - t0
    print(f"\n{nb_filename}  rc={rc}  elapsed={elapsed/60:.1f} min")
    if rc != 0:
        raise RuntimeError(f"{nb_filename} failed with return code {rc}")


def sync_to_drive():
    """Rsync local artifacts back to Drive so a disconnect doesn't lose progress."""
    SYNC_TARGETS = [
        ("models",                "models"),
        ("data/processed",        "data/processed"),
        ("supplementary/figures", "supplementary/figures"),
        ("notebooks",             "notebooks_executed"),   # snapshot of executed notebooks
        ("logs",                  "logs"),
    ]
    for local_rel, drive_rel in SYNC_TARGETS:
        local_path = COLAB_REPO_ROOT / local_rel
        drive_path = DRIVE_PROJECT_ROOT / drive_rel
        if not local_path.exists():
            continue
        drive_path.mkdir(parents=True, exist_ok=True)
        # rsync -a: preserve attrs; --delete keeps Drive in sync with local (removes
        # files on Drive that no longer exist locally, scoped to this dir)
        r = subprocess.run(
            ["rsync", "-a", f"{local_path}/", f"{drive_path}/"],
            capture_output=True, text=True,
        )
        if r.returncode != 0:
            print(f"  WARNING: rsync {local_rel} → {drive_rel} failed: {r.stderr[:300]}")
        else:
            print(f"  synced: {local_rel} → Drive/{drive_rel}")


def run_and_sync(nb_filename: str, timeout_sec: int = 1800):
    run_notebook(nb_filename, timeout_sec=timeout_sec)
    print("\nSyncing artifacts to Drive...")
    sync_to_drive()


## 7. nb01 — data collection + features (~5 min)

In [ ]:
run_and_sync("01_data_collection.ipynb", timeout_sec=600)

## 8. nb02 — EDA + normality (~2 min)

In [ ]:
run_and_sync("02_eda_normality.ipynb", timeout_sec=300)

## 9. nb03 — HMM training × 3 variants (~20-30 min)

In [ ]:
run_and_sync("03_hmm_regime.ipynb", timeout_sec=2400)

## 10. nb04 — baseline LSTMs × 4 variants × 3 seeds (~60-90 min)

Heaviest notebook. Each variant: seed 42 full Optuna, seeds 43/44 via `--fixed-hparams`. If this cell times out, bump the timeout and rerun — artifacts from earlier-finished variants are already on Drive.


In [ ]:
run_and_sync("04_lstm_baseline.ipynb", timeout_sec=7200)

## 11. nb05 — regime LSTMs × 3 variants × 3 seeds (~90-120 min)

Second-heaviest. Each variant invocation trains both regimes (calm + volatile) back-to-back per seed.


In [ ]:
run_and_sync("05_lstm_regime.ipynb", timeout_sec=10800)

## 12. nb06 — per-variant ensemble + within-variant DM (~5-10 min)

In [ ]:
run_and_sync("06_ensemble_eval.ipynb", timeout_sec=1200)

## 13. nb07 — cross-variant 6-way comparison + HAR-RV + DM + F1/F2 (~5 min)

In [ ]:
run_and_sync("07_variant_comparison.ipynb", timeout_sec=600)

## 14. Done — final artifact listing + summary

Below lists what was produced. Everything is already mirrored to Drive under `MyDrive/StockVolatilitySight/`.


In [ ]:
import os

def _list(path, max_lines=30):
    if not path.exists():
        return [f"(missing: {path})"]
    files = sorted(path.rglob("*"))
    files = [f for f in files if f.is_file()]
    out = []
    for f in files[:max_lines]:
        rel = f.relative_to(path)
        out.append(f"  {rel}  ({f.stat().st_size / 1024:.0f} KB)")
    if len(files) > max_lines:
        out.append(f"  ... and {len(files) - max_lines} more")
    return out


print("──── models/ ────")
for line in _list(COLAB_REPO_ROOT / "models"):
    print(line)
print()
print("──── data/processed/ (excluding seeds/) ────")
data_proc = COLAB_REPO_ROOT / "data" / "processed"
for f in sorted(data_proc.glob("*.parquet")):
    print(f"  {f.name}  ({f.stat().st_size / 1024:.0f} KB)")
print()
print("──── data/processed/seeds/ (per-seed predictions) ────")
for f in sorted((data_proc / "seeds").glob("*.parquet"))[:20]:
    print(f"  seeds/{f.name}")
print()
print("──── supplementary/figures/ ────")
for line in _list(COLAB_REPO_ROOT / "supplementary" / "figures"):
    print(line)
print()
print("Drive mirror: /content/drive/MyDrive/StockVolatilitySight/")
print("Done. Close the session after confirming the artifacts look right.")
